# Entrenar el modelo de EPP de DROID-E3D

Antes de empezar: **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)** y Guardar.

Después ejecutá cada celda en orden con el botón ▶ (esperá a que termine una antes de la siguiente).

In [ ]:
# 1. Instala YOLO (tarda 1 minuto)
!pip install -q ultralytics

In [ ]:
# 2. Subí DOS archivos cuando aparezca el botón «Elegir archivos»:
#    - modelo_epp.pt  (de src/SMAT/servidor_ia)
#    - el .zip que descargaste de Roboflow (formato YOLOv8)
from google.colab import files
subidos = files.upload()
print(list(subidos))

In [ ]:
# 3. Descomprime las fotos de Roboflow y las ORDENA:
#    - deja solo las 6 clases (lo marcado como "Lentes" pasa a Goggles; lo demás se descarta)
#    - si no hay fotos de validación, separa el 20 % para que la IA "se tome examen"
import glob, os, random, shutil, zipfile
zip_roboflow = [n for n in subidos if n.endswith('.zip')][0]
shutil.rmtree('fotos', ignore_errors=True)
zipfile.ZipFile(zip_roboflow).extractall('fotos')

import yaml
datos = yaml.safe_load(open('fotos/data.yaml'))
viejas = datos['names'] if isinstance(datos['names'], list) else [datos['names'][i] for i in sorted(datos['names'])]
CLASES = ['Hardhat', 'NO-Hardhat', 'Safety Vest', 'NO-Safety Vest', 'Goggles', 'NO-Goggles']
SINONIMOS = {'NO-Goggle': 'NO-Goggles', 'Lentes': 'Goggles', 'lentes': 'Goggles', 'Casco': 'Hardhat', 'Chaleco': 'Safety Vest'}
mapa = {}
for i, nombre in enumerate(viejas):
    nombre = SINONIMOS.get(nombre, nombre)
    if nombre in CLASES:
        mapa[i] = CLASES.index(nombre)
print('Clases en Roboflow:', viejas)
print('Se usan:', {viejas[i]: CLASES[j] for i, j in mapa.items()})

# Reescribe cada archivo de marcas con los números nuevos
for txt in glob.glob('fotos/**/labels/*.txt', recursive=True):
    lineas = []
    for linea in open(txt).read().splitlines():
        partes = linea.split()
        if partes and int(partes[0]) in mapa:
            lineas.append(' '.join([str(mapa[int(partes[0])])] + partes[1:]))
    open(txt, 'w').write('\n'.join(lineas))

# Si no hay validación, mueve el 20 % de train a valid
os.makedirs('fotos/valid/images', exist_ok=True); os.makedirs('fotos/valid/labels', exist_ok=True)
if not glob.glob('fotos/valid/images/*'):
    fotos_train = sorted(glob.glob('fotos/train/images/*'))
    random.seed(7); random.shuffle(fotos_train)
    for foto in fotos_train[: len(fotos_train) // 5]:
        base = os.path.splitext(os.path.basename(foto))[0]
        shutil.move(foto, 'fotos/valid/images/')
        etiqueta = f'fotos/train/labels/{base}.txt'
        if os.path.exists(etiqueta):
            shutil.move(etiqueta, 'fotos/valid/labels/')

ruta_fotos = os.path.abspath('fotos')
yaml.safe_dump({'path': ruta_fotos, 'train': 'train/images', 'val': 'valid/images', 'nc': len(CLASES), 'names': CLASES},
               open('fotos/data.yaml', 'w'), sort_keys=False)
print('Fotos para estudiar:', len(glob.glob('fotos/train/images/*')), '· para la prueba:', len(glob.glob('fotos/valid/images/*')))


In [ ]:
# 4. ENTRENA (20 a 40 minutos). Parte de lo que el modelo ya sabe y aprende de NUESTRAS fotos.
#    imgsz=416 es el mismo tamaño que usa motor_ia_robot.py
from ultralytics import YOLO
modelo = YOLO('modelo_epp.pt')
modelo.train(data='fotos/data.yaml', epochs=60, imgsz=416, batch=16, patience=15, name='droid')

In [ ]:
# 5. Descarga el modelo nuevo (se llama modelo_epp_nuevo.pt)
#    Busca el último best.pt, aunque YOLO lo haya guardado en droid2, train, etc.
import glob, shutil
from google.colab import files
encontrados = sorted(glob.glob('/content/**/weights/best.pt', recursive=True))
print(encontrados)
if not encontrados:
    print('No está el modelo: Colab se reinició. Volvé a correr las celdas 1 a 4.')
else:
    shutil.copy(encontrados[-1], 'modelo_epp_nuevo.pt')
    files.download('modelo_epp_nuevo.pt')
